# Phase 5: Ensemble Learning

## Objective
Compare tree-based ensemble models with traditional regression
methods for day-ahead electricity demand forecasting.

## Models
- Random Forest Regression
- XGBoost Regression

## Evaluation
- Training: 2021–2023
- Validation: 2024
- Final Test: 2025 (reserved)

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error

project = Path.cwd()

if not (project / "data_processed").exists():
    project = project.parent

ml_df = pd.read_csv(
    project / "data_processed" / "pjm_rto_ml_features.csv"
)

ml_df["forecast_origin"] = pd.to_datetime(
    ml_df["forecast_origin"], utc=True
)

ml_df["forecast_year"] = (
    ml_df["forecast_origin"]
    .dt.tz_convert("America/New_York")
    .dt.year
)

train = ml_df[ml_df["forecast_year"] <= 2023].copy()
val = ml_df[ml_df["forecast_year"] == 2024].copy()

features = [
    "hour", "day_of_week", "month", "is_weekend", "horizon",
    "lag_24", "lag_48", "lag_168",
    "rolling_mean_24", "rolling_std_24",
    "rolling_mean_168", "rolling_std_168"
]

X_train = train[features]
y_train = train["mw"]

X_val = val[features]
y_val = val["mw"]

print("Training:", X_train.shape)
print("Validation:", X_val.shape)

Training: (26112, 12)
Validation: (8784, 12)


In [2]:
# Step 5.1: Random Forest Regression

rf_model = RandomForestRegressor(
    n_estimators=200,
    max_depth=15,
    min_samples_leaf=3,
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train, y_train)

rf_pred = rf_model.predict(X_val)

rf_rmse = np.sqrt(mean_squared_error(y_val, rf_pred))
rf_mae = mean_absolute_error(y_val, rf_pred)

print("Random Forest — Validation 2024")
print(f"RMSE: {rf_rmse:,.2f} MW")
print(f"MAE:  {rf_mae:,.2f} MW")

Random Forest — Validation 2024
RMSE: 5,410.38 MW
MAE:  3,931.35 MW


In [4]:
baseline_results = pd.DataFrame([
    {"Model": "Seasonal Naive", "RMSE": 6422.94, "MAE": 4799.88},
    {"Model": "Linear Regression", "RMSE": 5660.35, "MAE": 4295.51},
    {"Model": "Ridge Regression", "RMSE": 5660.19, "MAE": 4295.42},
    {"Model": "Lasso Regression", "RMSE": 5658.35, "MAE": 4293.93},
    {"Model": "Random Forest", "RMSE": rf_rmse, "MAE": rf_mae}
])

display(baseline_results.sort_values("RMSE").round(2))

,Model,RMSE,MAE
4,Random Forest,5410.38,3931.35
3,Lasso Regression,5658.35,4293.93
2,Ridge Regression,5660.19,4295.42
1,Linear Regression,5660.35,4295.51
0,Seasonal Naive,6422.94,4799.88


In [6]:
# Step 5.2: XGBoost Regression

from xgboost import XGBRegressor

xgb_model = XGBRegressor(
    n_estimators=500,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="reg:squarederror",
    random_state=42,
    n_jobs=-1
)

# Train model
xgb_model.fit(X_train, y_train)

# Predict validation set
xgb_pred = xgb_model.predict(X_val)

# Evaluate performance
xgb_rmse = np.sqrt(mean_squared_error(y_val, xgb_pred))
xgb_mae = mean_absolute_error(y_val, xgb_pred)

print("XGBoost — Validation 2024")
print(f"RMSE: {xgb_rmse:,.2f} MW")
print(f"MAE:  {xgb_mae:,.2f} MW")

XGBoost — Validation 2024
RMSE: 5,288.81 MW
MAE:  3,824.08 MW


In [7]:
# Step 5.2: Update model comparison

xgb_result = pd.DataFrame([{
    "Model": "XGBoost",
    "RMSE": xgb_rmse,
    "MAE": xgb_mae
}])

baseline_results = pd.concat(
    [baseline_results, xgb_result],
    ignore_index=True
)

display(
    baseline_results
    .sort_values("RMSE")
    .round(2)
)

,Model,RMSE,MAE
5,XGBoost,5288.81,3824.08
4,Random Forest,5410.38,3931.35
3,Lasso Regression,5658.35,4293.93
2,Ridge Regression,5660.19,4295.42
1,Linear Regression,5660.35,4295.51
0,Seasonal Naive,6422.94,4799.88
